# Детекция фейковых новостей с DeepSeek API

В этом ноутбуке реализована детекция фейковых новостей с помощью **DeepSeek API** в режиме zero-shot.

Для каждой новости модель возвращает два поля:
- `label` — `1` (правда) / `0` (фейк)  
- `reason` — краткое объяснение решения

Результаты сохраняются в `models/deepseek/` и сравниваются с другими подходами.

## 1. Установка зависимостей

In [ ]:
!pip install openai scikit-learn pandas matplotlib seaborn tqdm -q

## 2. Импорты и конфигурация

In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

import warnings

warnings.filterwarnings("ignore")

# Загружаем .env из корня проекта
load_dotenv(Path("../../.env").resolve())

SEED = 42
np.random.seed(SEED)

In [ ]:
@dataclass
class Config:
    # Данные
    data_path: str = "../../data/ready_dataset.csv"
    text_column: str = "combined_text"
    label_column: str = "label"
    test_size: float = 0.2
    sample_size: int = 100

    # DeepSeek API (ключ берётся из .env)
    api_key: str = field(default_factory=lambda: os.environ["DEEPSEEK_API_KEY"])
    base_url: str = "https://api.deepseek.com"
    model: str = "deepseek-chat"
    max_tokens: int = 512
    temperature: float = 0.0
    request_delay: float = 0.3
    max_retries: int = 3

    # Сохранение
    output_dir: str = "../../models/deepseek"


cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

print("Config готов. Output dir:", cfg.output_dir)
print(f"Model: {cfg.model}")
print(f"API key: {cfg.api_key[:8]}…{cfg.api_key[-4:]} (len={len(cfg.api_key)})")

## 3. Загрузка данных

In [ ]:
df = pd.read_csv(cfg.data_path)
df = df[[cfg.text_column, cfg.label_column]].dropna()
df[cfg.text_column] = df[cfg.text_column].astype(str).str.strip()
df = df[df[cfg.text_column] != ""]
df[cfg.label_column] = pd.to_numeric(df[cfg.label_column], errors="coerce")
df = df.dropna(subset=[cfg.label_column])
df[cfg.label_column] = df[cfg.label_column].astype(int)

print(f"Всего записей: {len(df)}")
print(
    f'Распределение меток:\n{df[cfg.label_column].value_counts().rename({0: "Фейк (0)", 1: "Правда (1)"})}'
)

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=cfg.test_size, random_state=SEED, stratify=df[cfg.label_column]
)

if cfg.sample_size and cfg.sample_size < len(test_df):
    test_df = test_df.sample(cfg.sample_size, random_state=SEED)

test_df = test_df.reset_index(drop=True)
print(f"Тестовая выборка: {len(test_df)} примеров")
print(
    f"Фейк: {(test_df[cfg.label_column] == 0).sum()}, Правда: {(test_df[cfg.label_column] == 1).sum()}"
)

# Few-shot пул: 10 фейков + 10 правды из TRAIN (не test!)
N_FEWSHOT_PER_CLASS = 10
fewshot_fake = train_df[train_df[cfg.label_column] == 0].sample(
    N_FEWSHOT_PER_CLASS, random_state=SEED
)
fewshot_real = train_df[train_df[cfg.label_column] == 1].sample(
    N_FEWSHOT_PER_CLASS, random_state=SEED
)

fewshot_examples = []
for _, r in fewshot_fake.iterrows():
    fewshot_examples.append({"text": r[cfg.text_column], "label": 0})
for _, r in fewshot_real.iterrows():
    fewshot_examples.append({"text": r[cfg.text_column], "label": 1})

import random

random.Random(SEED).shuffle(fewshot_examples)
print(f"Few-shot примеров: {len(fewshot_examples)} (из train, не пересекается с test)")

## 4. DeepSeek API клиент

In [ ]:
class DeepSeekClient:
    def __init__(
        self,
        api_key: str,
        base_url: str,
        model: str,
        max_tokens: int = 256,
        temperature: float = 0.0,
        fewshot_examples: list = None,
    ):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.fewshot = fewshot_examples or []

        self.system_prompt = (
            "You are an expert classifier for a Russian-language news dataset. "
            "The texts are LEMMATIZED Russian news articles (no punctuation, no original headlines, "
            "no source attribution). Your task: classify each text as REAL (label=1) or FAKE (label=0).\n\n"
            "IMPORTANT context about this dataset:\n"
            "- Stylistic markers (clickbait, ALL CAPS, punctuation) are USELESS — preprocessing stripped them\n"
            "- Both fakes and real news mention named officials, dates, institutions in neutral tone\n"
            "- FAKE in this dataset: political satire, fabricated quotes from politicians "
            "(Putin, Trump, Peskov, Zelensky, Maduro, Lukashenko, Yanukovych, Lyashko, etc.), "
            "absurd-but-plausible-sounding government decisions, mock pro-Kremlin propaganda, "
            'fake celebrity political moves, satirical "announcements" about parties / movements / '
            "officials, fake quotes attributed to ministries, fabricated bureaucratic absurdity, "
            'invented "patriotic" or "anti-corruption" initiatives that sound like parody\n'
            "- REAL in this dataset: factual reporting on war events (Ukraine, mobilization, casualties, "
            "attacks, evacuations), sanctions, official statements about military operations, "
            "court rulings, criminal cases against opposition figures, refugee movements, "
            "documented atrocities, missile strikes, foreign policy moves, economic data\n\n"
            "You will receive 20 LABELED examples first. Study them — they reflect the EXACT "
            "distribution of this dataset. Then classify the final query using the same rubric.\n\n"
            "DECISION RULES:\n"
            "- If the article describes something that sounds like satirical/absurd political theater "
            "(e.g. politician dressing up, fake party, mock movement, parody decree) → label=0 (FAKE)\n"
            "- If the article describes documented war/legal/diplomatic/economic events → label=1 (REAL)\n"
            '- For ambiguous political quotes: ask "could this be a Panorama-style fabrication?" — if yes, label=0\n'
            "- If completely unsure: prefer the label of the MOST SIMILAR example you saw above\n\n"
            "RESPONSE FORMAT — strictly one JSON object, no markdown, no preamble:\n"
            '{"label": 1 or 0, "reason": "one short sentence"}\n\n'
            "IMPORTANT: Always return JSON. Match the patterns from the 20 labeled examples."
        )

    def _build_messages(self, text: str) -> list:
        messages = [{"role": "system", "content": self.system_prompt}]
        for ex in self.fewshot:
            messages.append(
                {
                    "role": "user",
                    "content": f'TEXT:\n{ex["text"][:1500]}\n\nClassify (return JSON):',
                }
            )
            reason = (
                "Matches fake patterns from this dataset (political satire / fabricated political claim)."
                if ex["label"] == 0
                else "Matches real patterns from this dataset (factual reporting on documented events)."
            )
            messages.append(
                {
                    "role": "assistant",
                    "content": json.dumps(
                        {"label": ex["label"], "reason": reason}, ensure_ascii=False
                    ),
                }
            )
        messages.append(
            {
                "role": "user",
                "content": f"TEXT:\n{text[:1500]}\n\nClassify (return JSON):",
            }
        )
        return messages

    def classify(self, text: str) -> dict:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self._build_messages(text),
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            stream=False,
        )
        content = response.choices[0].message.content.strip()
        return self._parse_response(content)

    def classify_self_consistency(self, text: str, n: int = 5) -> dict:
        """Self-consistency: голосование большинством по N сэмплам с T>0."""
        votes_true = 0
        votes_false = 0
        explanations = []
        for i in range(n):
            # Первый сэмпл — детерминированный (T=0), остальные — с разнообразием
            T = 0.0 if i == 0 else 0.6
            response = self.client.chat.completions.create(
                model=self.model,
                messages=self._build_messages(text),
                max_tokens=self.max_tokens,
                temperature=T,
                stream=False,
            )
            content = response.choices[0].message.content.strip()
            parsed = self._parse_response(content)
            if parsed["is_true"] is True:
                votes_true += 1
                explanations.append(parsed["explanation"])
            elif parsed["is_true"] is False:
                votes_false += 1
                explanations.append(parsed["explanation"])
        if votes_true == 0 and votes_false == 0:
            return {"is_true": None, "explanation": "no valid votes"}
        is_true = votes_true > votes_false
        return {
            "is_true": is_true,
            "explanation": f"votes real={votes_true} fake={votes_false}. "
            + (explanations[0] if explanations else ""),
        }

    @staticmethod
    def _parse_response(content: str) -> dict:
        original = content
        content = content.strip()

        if "```" in content:
            parts = content.split("```")
            for part in parts:
                part = part.strip()
                if part.startswith("json"):
                    part = part[4:].strip()
                if part.startswith("{"):
                    content = part
                    break

        matches = re.findall(r"\{[^{}]*\}", content, re.DOTALL)
        if matches:
            content = matches[-1]

        try:
            data = json.loads(content)
            label_key = next(
                (
                    k
                    for k in data
                    if k.lower() in ("label", "is_true", "is_real", "class", "result")
                ),
                None,
            )
            reason_key = next(
                (
                    k
                    for k in data
                    if any(s in k.lower() for s in ("reason", "explan", "comment"))
                ),
                None,
            )
            if label_key is not None:
                raw = data[label_key]
                if isinstance(raw, bool):
                    is_true = raw
                elif isinstance(raw, (int, float)):
                    is_true = bool(int(raw))
                else:
                    s = str(raw).strip().lower()
                    is_true = s in ("1", "true", "real", "правда")
                explanation = str(data[reason_key]).strip() if reason_key else original
                return {"is_true": is_true, "explanation": explanation}
        except Exception:
            pass

        lower = original.lower()
        label_matches = re.findall(r'"label"\s*:\s*([01])', lower)
        if label_matches:
            return {"is_true": label_matches[-1] == "1", "explanation": original}
        label_matches = re.findall(r'"label"\s*:\s*(true|false)', lower)
        if label_matches:
            return {"is_true": label_matches[-1] == "true", "explanation": original}

        if any(
            w in lower
            for w in ("real news", "credible", "verified", "legitimate", "правд")
        ):
            return {"is_true": True, "explanation": original}
        if any(
            w in lower
            for w in ("fake news", "fabricat", "misinform", "mislead", "фейк")
        ):
            return {"is_true": False, "explanation": original}

        return {"is_true": None, "explanation": original}


client = DeepSeekClient(
    api_key=cfg.api_key,
    base_url=cfg.base_url,
    model=cfg.model,
    max_tokens=cfg.max_tokens,
    temperature=cfg.temperature,
    fewshot_examples=fewshot_examples,
)
print(f"Клиент создан, few-shot примеров: {len(client.fewshot)}")

## 5. Проверка одного запроса

In [ ]:
sample_text = test_df[cfg.text_column].iloc[0]
sample_label = test_df[cfg.label_column].iloc[0]

print(f"Текст (первые 300 символов):\n{sample_text[:300]}\n")
print(f'Истинная метка: {"Фейк" if sample_label == 0 else "Правда"} ({sample_label})')
print("\nЗапрос к DeepSeek...")

result = client.classify(sample_text)
print(f"\nОтвет модели:")
print(f'  is_true:     {result["is_true"]}')
print(f'  explanation: {result["explanation"]}')

## 6. Классификация тестовой выборки

In [ ]:
results = []
errors = []

# Используем self-consistency: 5 сэмплов на пример, голосование большинством.
# Стоимость x5, но на пограничных кейсах даёт +1-3% accuracy.
USE_SELF_CONSISTENCY = True
N_VOTES = 5

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="DeepSeek classify"):
    text = row[cfg.text_column]
    label = row[cfg.label_column]

    prediction = {"is_true": None, "explanation": ""}
    last_error = None
    raw_attempts = []

    for attempt in range(cfg.max_retries):
        try:
            if USE_SELF_CONSISTENCY:
                pred = client.classify_self_consistency(text, n=N_VOTES)
            else:
                pred = client.classify(text)
            raw_attempts.append(pred["explanation"])
            if pred["is_true"] is not None:
                prediction = pred
                break
            prediction = pred
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            raw_attempts.append(f"[ERROR] {last_error}")
        time.sleep(cfg.request_delay)

    if last_error and prediction["is_true"] is None:
        errors.append({"idx": idx, "error": last_error})

    results.append(
        {
            "idx": idx,
            "text": text[:300],
            "true_label": int(label),
            "is_true": prediction["is_true"],
            "explanation": prediction["explanation"] or (last_error or ""),
            "attempts": len(raw_attempts),
            "error": last_error,
        }
    )

    time.sleep(cfg.request_delay)

results_df = pd.DataFrame(results)
n_none = results_df["is_true"].isna().sum()
print(f"\nГотово.")
print(f"  Всего обработано:      {len(results_df)}")
print(f"  Ошибок API:            {len(errors)}")
print(f"  Не дали ответа:        {n_none}")
print(f'  Среднее число попыток: {results_df["attempts"].mean():.2f}')
print(
    f'  Метод:                 {"self-consistency n=" + str(N_VOTES) if USE_SELF_CONSISTENCY else "single-shot"}'
)

if errors:
    print(f"\nПервые 3 ошибки API:")
    for e in errors[:3]:
        print(f'  idx={e["idx"]}: {e["error"]}')

In [ ]:
results_path = Path(cfg.output_dir) / "predictions.csv"
results_df.to_csv(results_path, index=False, encoding="utf-8")
print(f"Сохранено: {results_path}")

results_df[["true_label", "is_true", "explanation"]].head(5)

## 7. Оценка качества

In [ ]:
eval_df = results_df.copy()


def to_pred(row):
    if pd.isna(row["is_true"]) or row["is_true"] is None:
        return 1 - int(row["true_label"])
    return int(bool(row["is_true"]))


eval_df["y_pred"] = eval_df.apply(to_pred, axis=1)

y_pred = eval_df["y_pred"].values
y_true = eval_df["true_label"].astype(int).values

print(f"Всего в оценке: {len(eval_df)}")
print(f'  с валидным ответом:    {eval_df["is_true"].notna().sum()}')
print(f'  нераспаршенных (=err): {eval_df["is_true"].isna().sum()}')

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")
prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)

try:
    auc = roc_auc_score(y_true, y_pred)
except Exception:
    auc = float("nan")

metrics = {
    "accuracy": round(acc, 4),
    "f1": round(f1, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "roc_auc": round(auc, 4),
    "n_total": int(len(eval_df)),
    "n_valid": int(eval_df["is_true"].notna().sum()),
}

print()
for k, v in metrics.items():
    print(f"  {k:12s}: {v}")

In [ ]:
metrics_path = Path(cfg.output_dir) / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f"Метрики сохранены: {metrics_path}")

## 8. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("DeepSeek — детекция фейковых новостей", fontsize=14, fontweight="bold")

# --- Confusion matrix ---
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    ax=axes[0],
    xticklabels=["Фейк (0)", "Правда (1)"],
    yticklabels=["Фейк (0)", "Правда (1)"],
)
axes[0].set_title("Матрица ошибок")
axes[0].set_ylabel("Истинная метка")
axes[0].set_xlabel("Предсказание")

# --- Метрики ---
metric_names = ["Accuracy", "F1", "Precision", "Recall"]
metric_values = [acc, f1, prec, rec]
colors = ["#27AE60", "#2ECC71", "#F39C12", "#E74C3C"]

bars = axes[1].bar(metric_names, metric_values, color=colors, width=0.5)
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Метрики качества")
axes[1].set_ylabel("Значение")

for bar, val in zip(bars, metric_values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

plt.tight_layout()
plot_path = Path(cfg.output_dir) / "metrics_plot.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"График сохранён: {plot_path}")

## 9. Примеры предсказаний

In [ ]:
def label_name(val):
    return "Правда" if val == 1 else "Фейк"


valid_eval = eval_df.dropna(subset=["is_true"]).copy()
yt = valid_eval["true_label"].astype(int).values
yp = valid_eval["is_true"].astype(bool).astype(int).values

correct = valid_eval[yt == yp].sample(min(3, (yt == yp).sum()), random_state=SEED)
print("=== Верные предсказания ===")
for _, row in correct.iterrows():
    print(f'  Метка: {label_name(int(row["true_label"]))}')
    print(f'  Объяснение: {row["explanation"][:200]}')
    print()

wrong = valid_eval[yt != yp]
if len(wrong):
    wrong_sample = wrong.sample(min(3, len(wrong)), random_state=SEED)
    print("=== Ошибки ===")
    for _, row in wrong_sample.iterrows():
        pred_label = int(row["is_true"])
        print(
            f'  Истинная: {label_name(int(row["true_label"]))}, Предсказана: {label_name(pred_label)}'
        )
        print(f'  Объяснение: {row["explanation"][:200]}')
        print()

## 10. Classification report

In [ ]:
print(
    classification_report(
        y_true, y_pred, target_names=["Фейк (0)", "Правда (1)"], digits=4
    )
)

## 11. Итог

| Метрика    | Значение |
|------------|----------|
| Accuracy   | см. выше |
| F1         | см. выше |
| Precision  | см. выше |
| Recall     | см. выше |

**Выводы:**
- DeepSeek в режиме zero-shot классифицирует новости без какого-либо обучения.
- Прямой промпт (детекция фейков без обходных манёвров) работает благодаря отсутствию жёстких ограничений цензора.
- Модель возвращает структурированный JSON с полями `label` и `reason`.
- Результаты сохраняются в `models/deepseek/` для сравнения с GigaChat и другими подходами.

## 12. Все ответы модели

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

view_df = results_df[
    ["idx", "true_label", "is_true", "attempts", "text", "explanation"]
].copy()
view_df["true_label"] = view_df["true_label"].map({0: "Фейк", 1: "Правда"})
view_df["is_true"] = view_df["is_true"].map(
    {True: "Правда", False: "Фейк", None: "— нет ответа —"}
)

log_path = Path(cfg.output_dir) / "all_responses.csv"
view_df.to_csv(log_path, index=False, encoding="utf-8")
print(f"Полный лог сохранён: {log_path}\n")

view_df